In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F


Net = nn.Sequential(nn.Linear(20,4), nn.ReLU(), nn.Linear(4,1))
X = torch.rand(64,20)
Net(X)
print(Net[2].state_dict())

print(Net[2].bias.grad)

print(Net)

OrderedDict([('weight', tensor([[ 0.2073,  0.1836,  0.3562, -0.3045]])), ('bias', tensor([-0.2219]))])
None
Sequential(
  (0): Linear(in_features=20, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)


In [26]:
print(Net[2].weight)
print()
print(Net[2].weight.grad)

Parameter containing:
tensor([[ 0.2073,  0.1836,  0.3562, -0.3045]], requires_grad=True)

None


每个参数都表示为参数类的一个实例，是复合的对象：包含值、梯度和额外信息。

In [27]:
for name, param in Net.named_parameters():
    print(f"参数名字: {name}, 形状: {param.shape}")
    print(param)

参数名字: 0.weight, 形状: torch.Size([4, 20])
Parameter containing:
tensor([[ 0.0105,  0.0796,  0.2113, -0.1384, -0.2139,  0.0716, -0.0600, -0.1046,
          0.0626,  0.0068,  0.2183,  0.1572,  0.1000,  0.1800, -0.1536, -0.1216,
         -0.1070,  0.0150, -0.1443, -0.1321],
        [-0.1174,  0.0256, -0.0261, -0.1229, -0.1608,  0.0066, -0.1168,  0.2104,
         -0.1653, -0.2217, -0.0628, -0.0330,  0.0480, -0.2179,  0.2193, -0.0381,
         -0.0181, -0.1176,  0.0828, -0.1032],
        [ 0.1588,  0.1291,  0.1192,  0.0276,  0.1890,  0.2004, -0.0020,  0.1758,
          0.0021, -0.2048, -0.1151,  0.2084, -0.1962,  0.0186,  0.0305, -0.0709,
          0.1471,  0.1169,  0.0671, -0.2213],
        [-0.1436,  0.0842, -0.1956, -0.0335, -0.0389,  0.1743, -0.1077, -0.2142,
         -0.0272,  0.0527, -0.1422, -0.1432,  0.1197,  0.0183,  0.2011,  0.0386,
         -0.1807, -0.1608, -0.1885, -0.1535]], requires_grad=True)
参数名字: 0.bias, 形状: torch.Size([4])
Parameter containing:
tensor([-0.0744,  0.0418,  0.

In [28]:
def block1():
    return nn.Sequential(nn.Linear(2,4), nn.ReLU(), nn.Linear(4,2), nn.ReLU())


def block2():
    Net = nn.Sequential()
    for i in range(4):
        Net.add_module(f'block{i}',block1())
    return Net

BigNet = nn.Sequential(block2(), nn.Linear(2,1))
print(BigNet)
# print(BigNet[0])

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=2, out_features=1, bias=True)
)


In [29]:
# print(BigNet[0])
print(BigNet[1])

Linear(in_features=2, out_features=1, bias=True)


In [30]:
print(BigNet[0][2])

Sequential(
  (0): Linear(in_features=2, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
)


In [31]:
print(BigNet[0][2][2].weight)

Parameter containing:
tensor([[-0.3507, -0.0891,  0.4126,  0.0773],
        [ 0.4710, -0.3775, -0.0463, -0.3071]], requires_grad=True)


In [32]:
def init_params(block):
    if type(block) == nn.Linear:
        nn.init.normal_(block.weight, 0, 0.01)
        nn.init.normal_(block.bias, 0, 0.01)

BigNet.apply(init_params)
BigNet[0][0][0].weight.data[0], BigNet[0][0][0].bias.data[0]

(tensor([ 1.1282e-05, -1.5777e-02]), tensor(0.0106))

### torch.nn.init
专门用于初始化参数的，
nn.init.uniform_(w, a=0, b=1)：均匀分布。
nn.init.zeros_(w) / nn.init.ones_(w)：全 0 或 全 1（通常用来初始化偏置 bias）。

针对特定激活函数设计的初始化：

nn.init.xavier_uniform_(w) / nn.init.xavier_normal_(w)： 也叫 Glorot 初始化。适合搭配 Sigmoid 或 Tanh 激活函数的网络，能有效缓解梯度消失。

nn.init.kaiming_uniform_(w) / nn.init.kaiming_normal_(w)： 也叫 He 初始化。专门为 ReLU 及其变体激活函数量身定制的初始化方法。

nn.init里的函数名最后都有一个下划线 _
在 PyTorch 中，结尾带 _ 的函数代表原地操作 (In-place)。意思是：它不创建新的内存空间，直接把传入的张量内部的数值给强行改写掉。


### 默认初始化
每一个内置的层都有一个隐藏的函数叫做 reset_parameters()。当你执行类似 self.hidden = nn.Linear(20, 256) 时，PyTorch 的 __init__ 函数最后一步就会自动调用reset_parameters()。
默认初始化使用了一种根据输入维度自适应的均匀分布。假设输入维度是 N，权重、偏置会被初始化为一个均匀分布。分布的范围是：$$\left[ -\frac{1}{\sqrt{N_{in}}}, \frac{1}{\sqrt{N_{in}}} \right]$$
目的是为了控制方差，防止数值爆炸或消失。输入连接的神经元越多（N越大），初始化的权重就应该越小。


### apply：PyTorch中的声明式编程
只需要定义对单个层要做什么，然后 .apply，将自动遍历。

In [33]:
def init_xavier(block):
    if type(block) == nn.Linear:
        nn.init.xavier_normal_(block.weight)

def init_42(block):
    if type(block) == nn.Linear:
        nn.init.constant_(block.weight, 42)

BigNet[0][0][0].apply(init_xavier)
BigNet[0][0][2].apply(init_42)

print(BigNet[0][0][0].weight)
print(BigNet[0][0][2].weight)

Parameter containing:
tensor([[-0.6156, -0.0621],
        [-0.0244, -0.4187],
        [-0.3731,  0.2041],
        [ 1.0391,  0.2822]], requires_grad=True)
Parameter containing:
tensor([[42., 42., 42., 42.],
        [42., 42., 42., 42.]], requires_grad=True)


In [34]:
BigNet[0][0][0].weight + 1
BigNet[0][0][0].weight.data[0][0] = 99
print(BigNet[0][0][0].weight)

Parameter containing:
tensor([[ 9.9000e+01, -6.2053e-02],
        [-2.4434e-02, -4.1872e-01],
        [-3.7312e-01,  2.0406e-01],
        [ 1.0391e+00,  2.8220e-01]], requires_grad=True)


### 共享参数
参数共享在CV，NLP，注意力等领域有很大作用，共享的参数的梯度最终会求和在一起

In [35]:
share_block = nn.Linear(3,8)
net = nn.Sequential(nn.Linear(10,3), nn.ReLU(), share_block, nn.ReLU(), nn.Linear(8,3), nn.ReLU(), share_block, nn.ReLU(), nn.Linear(8,2))

print(net[2].weight == net[6].weight)

tensor([[True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True]])
